# Support Vector Regression (SVR)

Support Vector Regression applies the principles of **Support Vector Machines** to regression problems. Instead of finding a hyperplane that separates classes, SVR finds a hyperplane (or curve for non-linear kernels) that fits the data within a **margin of tolerance (ε — epsilon)**.

### Key Idea — ε-Insensitive Tube
> SVR tries to fit as many data points as possible **within the ε-tube** around the regression line. Points outside the tube (support vectors) incur a penalty.

$$\text{Minimise: } \frac{1}{2}\|w\|^2 + C \sum (\xi_i + \xi_i^*)$$

| Parameter | Effect |
|-----------|--------|
| `C` | Regularisation — high C → less margin, fits data closer |
| `epsilon` | Width of the ε-tube — larger ε tolerates more error |
| `kernel` | `linear`, `rbf`, `poly` — controls the shape of the boundary |

---

## Why Diabetes Dataset?
> The **Diabetes** dataset has a moderate number of samples (442) and 10 well-scaled features, making SVR computationally feasible. The target shows **non-linear patterns** with respect to individual features, which the RBF kernel of SVR handles effectively. This makes it ideal for comparing SVR kernels and demonstrating the ε-tube concept.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling is ESSENTIAL for SVR
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_s = scaler_X.fit_transform(X_train)
X_test_s  = scaler_X.transform(X_test)
y_train_s = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

# Compare kernels
results = {}
for kernel in ['linear', 'rbf', 'poly']:
    svr = SVR(kernel=kernel, C=100, epsilon=0.1)
    svr.fit(X_train_s, y_train_s)
    y_pred_s = svr.predict(X_test_s)
    y_pred = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
    results[kernel] = {'model': svr, 'y_pred': y_pred}
    print(f'Kernel={kernel:<8}  R²={r2_score(y_test, y_pred):.4f}  RMSE={np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')

In [ ]:
# Actual vs Predicted for RBF kernel
y_pred_rbf = results['rbf']['y_pred']

plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_rbf, alpha=0.5, color='purple', edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel('Actual Disease Progression')
plt.ylabel('Predicted')
plt.title('SVR (RBF Kernel) — Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

# Residuals plot
residuals = y_test - y_pred_rbf
plt.figure(figsize=(8, 4))
plt.scatter(y_pred_rbf, residuals, alpha=0.5, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residual')
plt.title('SVR — Residual Plot')
plt.tight_layout()
plt.show()